In [ ]:
# Setup: clone the StarX repo and the pinned TripoSR commit, install this
# notebook's dependencies, prepare headless rendering, mount Drive.
import os
import subprocess
import sys

BRANCH = "main"
TRIPOSR_COMMIT = "107cefdc244c39106fa830359024f6a2f1c78871"
NOTEBOOK_ID = "06"

IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    REPO_DIR, TRIPOSR_DIR = "/content/StarX", "/content/TripoSR"
    if not os.path.exists(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--branch", BRANCH,
             "https://github.com/SattamAltwaim/StarX.git", REPO_DIR],
            check=True,
        )
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
    TRIPOSR_DIR = os.path.join(REPO_DIR, "third_party", "TripoSR")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if not os.path.exists(TRIPOSR_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/VAST-AI-Research/TripoSR.git",
         TRIPOSR_DIR],
        check=True,
    )
subprocess.run(["git", "-C", TRIPOSR_DIR, "checkout", "-q", TRIPOSR_COMMIT], check=True)

from starx import pins

assert pins.TRIPOSR_COMMIT == TRIPOSR_COMMIT, "notebook pin out of sync with starx/pins.py"
if pins.PIP_PINS[NOTEBOOK_ID]:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *pins.PIP_PINS[NOTEBOOK_ID]],
        check=True,
    )
# remove Colab preinstalls that break the pinned stack (see starx/pins.py)
if pins.PIP_UNINSTALL.get(NOTEBOOK_ID):
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-q", "-y",
         *pins.PIP_UNINSTALL[NOTEBOOK_ID]],
        check=False, capture_output=True,
    )

# headless rendering (for scoring and galleries), as in notebook 03
if IN_COLAB:
    subprocess.run(
        ["apt-get", "install", "-y", "-q", "libosmesa6"],
        check=False, capture_output=True,
    )
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

from starx import colab as scolab

DRIVE = scolab.mount_drive()
report = scolab.setup_report()

# 06 - Evaluation

Time to measure. On the held-out test split - designs the model never saw - this notebook scores every reconstruction three ways:

- **Chamfer distance**: average gap between the predicted surface and the true surface (lower is better);
- **F-score**: the fraction of surface that lands within a tolerance of the other surface, at tight/medium/loose thresholds (higher is better);
- **volumetric IoU**: how much the two solid volumes overlap (higher is better);

plus image metrics (PSNR, SSIM, LPIPS) on renders at held-out cameras. As the comparison anchor, the pretrained RGB TripoSR gets the dataset's own thumbnail of each part - a photo-versus-sketches contest.

Everything long here is resumable, and every number comes with the pictures that explain it: per-design walkthroughs, quality-sorted galleries, turntables, and a failure board. Use a GPU runtime.

In [ ]:
# Configuration - every tunable for this notebook lives here.
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import trimesh
from PIL import Image
from tqdm.auto import tqdm

from starx import cameras, checkpoint, data, render_gt, shards, viz
from starx.config import CAMERA_DISTANCE, StarXConfig, run_dir, shard_dir

SMOKE = False    # True: evaluate the smoke run on the smoke shards
RUN_NAME = "smoke_overfit" if SMOKE else "baseline_l4"
CKPT_STEP = None          # None: newest checkpoint of the run
EVAL_LIMIT = 20 if SMOKE else 300   # designs to evaluate (raise for the full split)
N_EVAL_VIEWS = 4          # stored GT views used for 2D metrics
MC_RES = 128 if SMOKE else 256      # marching-cubes resolution
BASELINE_THUMBNAIL = True # also evaluate pretrained TripoSR on thumbnails
SEED = 1337
SPLIT_PREFIX = "smoke_" if SMOKE else ""

cfg = StarXConfig(
    drive_root=str(DRIVE / "StarX")
    if DRIVE is not None
    else os.path.join(REPO_DIR, "data", "StarX"),
    local_root="/content/starx_local"
    if IN_COLAB
    else os.path.join(REPO_DIR, "data", "local"),
    # surgery - must match the trained run
    max_sketch_channels=6,
    conv_init="i3d_mean",
    lora_r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    dino_layers=(0, 1, 2, 3),
    train_triplane_embed=False,
    # evaluation
    mc_threshold=25.0,
    iou_res=64,
    n_points=10000,
    taus=(0.01, 0.02, 0.05),
    eval_chunk=131072,
    gt_size=256,
    seed=SEED,
)
print(f"evaluating run {RUN_NAME} on up to {EVAL_LIMIT} test designs")
print(f"3D metrics in normalized units (largest extent = 1); "
      f"f-score thresholds {cfg.taus}")

In [ ]:
# Trust the ruler before measuring: run every metric on shapes with known
# answers. A sphere against itself must score near-perfect; a sphere
# against a cube must score clearly worse.
sphere = trimesh.creation.icosphere(subdivisions=3, radius=0.5)
cube = trimesh.creation.box(extents=(1.0, 1.0, 1.0))
ps_a = seval.sample_surface_points(sphere, 4000, seed=0)
ps_b = seval.sample_surface_points(sphere, 4000, seed=1)
pc = seval.sample_surface_points(cube, 4000, seed=0)

cd_self = seval.chamfer_distance(ps_a, ps_b)
cd_cross = seval.chamfer_distance(ps_a, pc)
f_self = seval.fscore(ps_a, ps_b, tau=0.02)[0]
f_cross = seval.fscore(ps_a, pc, tau=0.02)[0]
grid_sphere = seval.occupancy_grid(ps_a, cfg.iou_res)
grid_cube = seval.occupancy_grid(pc, cfg.iou_res)
iou_self = seval.voxel_iou(grid_sphere, seval.occupancy_grid(ps_b, cfg.iou_res))
iou_cross = seval.voxel_iou(grid_sphere, grid_cube)

print(f"chamfer  sphere-sphere {cd_self:.4f}   sphere-cube {cd_cross:.4f}")
print(f"f@0.02   sphere-sphere {f_self:.3f}    sphere-cube {f_cross:.3f}")
print(f"iou      sphere-sphere {iou_self:.3f}    sphere-cube {iou_cross:.3f}")
assert cd_self < cd_cross and f_self > 0.9 and iou_self > iou_cross
print("metric sanity passed")

In [ ]:
# Load the trained model, the test shards, the zip (for ground-truth
# meshes and baseline thumbnails), and the offscreen renderer.
import torch

from starx import eval as seval
from starx import model as smodel

device = "cuda" if torch.cuda.is_available() else "cpu"
model, build_info = smodel.build_starx_model(cfg, TRIPOSR_DIR, device=device)
model.renderer.set_chunk_size(cfg.eval_chunk)
model.eval()

rdir = run_dir(cfg, RUN_NAME)
if CKPT_STEP is None:
    ckpt_path, ckpt_step = checkpoint.find_latest(rdir)
else:
    ckpt_step = CKPT_STEP
    ckpt_path = checkpoint.checkpoint_dir(rdir) / f"state_{CKPT_STEP:07d}.pt"
state = checkpoint.load_checkpoint(ckpt_path)
smodel.load_trainable_state_dict(model, state["model"])
print(f"loaded {RUN_NAME} checkpoint at step {ckpt_step}")

test_local = Path(cfg.local_root) / f"{SPLIT_PREFIX}test"
shards.prepare_local(shard_dir(cfg, f"{SPLIT_PREFIX}test"), test_local, progress=tqdm)
test_dataset = data.DesignDataset(test_local / "cache")
print(f"test designs available: {len(test_dataset)}")

zip_local_dir = "/content" if IN_COLAB else os.path.join(REPO_DIR, "data")
zip_path = scolab.ensure_zip_local(cfg, local_dir=zip_local_dir)
names = scolab.zip_inventory(zip_path)
design_ids_all = sorted(
    os.path.basename(n)[:-5]
    for n in names
    if n.endswith(".json") and "train_test" not in os.path.basename(n).lower()
)
zip_index = scolab.build_zip_index(names, design_ids_all)
WORK_DIR = Path("/content/work") if IN_COLAB else Path(REPO_DIR) / "data" / "work"

gt_renderer = render_gt.GTRenderer(cfg.gt_size)
print("renderer backend:", gt_renderer.backend)


def load_gt_mesh(design_id):
    """The design's final mesh, normalized exactly as in notebook 03."""
    member = zip_index[design_id].get("obj")
    if member is None:
        return None
    obj_path = scolab.extract_members(zip_path, [member], WORK_DIR)[0]
    mesh = trimesh.load(obj_path, force="mesh")
    os.remove(obj_path)
    return render_gt.normalize_mesh(mesh)[0]

In [ ]:
# Walk one design through the whole path: sketches -> scene code ->
# marching cubes -> interactive comparison against the ground-truth mesh.
walk_item = test_dataset[0]
fig = viz.show_sketch_stack(
    walk_item["stack_uint8"], walk_item["meta"], title=walk_item["design_id"]
)
plt.show()

with torch.no_grad():
    walk_code = smodel.encode_sketches(model, walk_item["sketch"][None].to(device))[0]
    walk_code = walk_code.float()
walk_pred_mesh = seval.extract_mesh(model, walk_code, res=MC_RES, threshold=cfg.mc_threshold)
walk_gt_mesh = load_gt_mesh(walk_item["design_id"])
assert walk_pred_mesh is not None, "no surface extracted - check the checkpoint"
print(f"prediction: {len(walk_pred_mesh.vertices)} vertices   "
      f"ground truth: {len(walk_gt_mesh.vertices)} vertices")
fig = viz.mesh_side_by_side(walk_gt_mesh, walk_pred_mesh, title=walk_item["design_id"])
fig.show()

In [ ]:
# The same design in 2D: render the reconstruction at stored ground-truth
# cameras (same renderer and light rig as the dataset) and score each view.
walk_pred_rgbs, _ = gt_renderer.render_mesh(
    walk_pred_mesh, walk_item["c2ws"][:N_EVAL_VIEWS]
)
fig, axes = plt.subplots(2, N_EVAL_VIEWS, figsize=(2.6 * N_EVAL_VIEWS, 5.4))
per_view = []
for v in range(N_EVAL_VIEWS):
    pred_t = torch.from_numpy(walk_pred_rgbs[v : v + 1]).float().permute(0, 3, 1, 2) / 255.0
    gt_t = torch.from_numpy(walk_item["views"][v : v + 1]).float().permute(0, 3, 1, 2) / 255.0
    m = seval.metrics_2d(pred_t, gt_t)
    per_view.append(m)
    axes[0, v].imshow(walk_item["views"][v])
    axes[0, v].set_title(f"ground truth {v}", fontsize=9)
    axes[1, v].imshow(walk_pred_rgbs[v])
    axes[1, v].set_title(f"psnr {m['psnr']:.1f}  ssim {m['ssim']:.2f}", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.show()
display(pd.DataFrame(per_view).round(3))

In [ ]:
# The evaluation loop: one CSV row per design, appended on Drive.
# Already-evaluated designs are skipped, so this cell resumes after any
# disconnect. 3D metrics compare surface samples of the reconstruction
# against the normalized ground-truth mesh; 2D metrics compare renders of
# the reconstruction against the stored ground-truth views.
results_path = rdir / "eval" / f"metrics_{SPLIT_PREFIX}test.csv"
results_path.parent.mkdir(parents=True, exist_ok=True)
done_ids = set(pd.read_csv(results_path)["design_id"]) if results_path.exists() else set()
print(f"{len(done_ids)} designs already evaluated")

for i in tqdm(range(min(EVAL_LIMIT, len(test_dataset))), desc="evaluating"):
    item = test_dataset[i]
    design_id = item["design_id"]
    if design_id in done_ids:
        continue
    row = {
        "design_id": design_id,
        "n_sketches": item["meta"]["n_sketches_total"],
        "truncated": item["meta"]["truncated"],
    }
    try:
        with torch.no_grad():
            code = smodel.encode_sketches(model, item["sketch"][None].to(device))[0]
            code = code.float()
        pred_mesh = seval.extract_mesh(model, code, res=MC_RES, threshold=cfg.mc_threshold)
        gt_mesh = load_gt_mesh(design_id)
        if pred_mesh is None or gt_mesh is None:
            row["failed"] = True
        else:
            p_gt = seval.sample_surface_points(gt_mesh, cfg.n_points, seed=SEED)
            p_pred = seval.sample_surface_points(pred_mesh, cfg.n_points, seed=SEED)
            row["chamfer"] = seval.chamfer_distance(p_gt, p_pred)
            for tau in cfg.taus:
                row[f"f@{tau}"] = seval.fscore(p_pred, p_gt, tau)[0]
            row["iou"] = seval.voxel_iou(
                seval.occupancy_grid(p_gt, cfg.iou_res),
                seval.occupancy_grid(p_pred, cfg.iou_res),
            )
            pred_rgbs, _ = gt_renderer.render_mesh(pred_mesh, item["c2ws"][:N_EVAL_VIEWS])
            pred_t = torch.from_numpy(pred_rgbs).float().permute(0, 3, 1, 2) / 255.0
            gt_t = (
                torch.from_numpy(item["views"][:N_EVAL_VIEWS]).float().permute(0, 3, 1, 2)
                / 255.0
            )
            row.update(seval.metrics_2d(pred_t, gt_t))
    except Exception as error:
        row["failed"] = True
        row["error"] = repr(error)
    pd.DataFrame([row]).to_csv(
        results_path, mode="a", header=not results_path.exists(), index=False
    )

In [ ]:
# The headline numbers, and how quality is distributed.
results_df = pd.read_csv(results_path)
ok_df = results_df[results_df.get("failed", pd.Series(dtype=bool)).fillna(False) == False].copy()
evaluated_ids = list(results_df["design_id"])

summary = ok_df[["chamfer"] + [f"f@{t}" for t in cfg.taus] + ["iou", "psnr", "ssim", "lpips"]]
display(summary.describe().T[["mean", "50%", "min", "max"]].rename(columns={"50%": "median"}))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
colors = plt.get_cmap("tab10").colors
for ax, metric, color in zip(axes, ["chamfer", "f@0.02", "iou"], colors):
    ax.hist(ok_df[metric].dropna(), bins=30, color=color)
    ax.set_xlabel(metric)
    ax.set_ylabel("designs")
fig.suptitle("metric distributions over the evaluated test designs")
fig.tight_layout()
plt.show()
print(f"evaluated: {len(results_df)}   with a valid mesh: {len(ok_df)}")

In [ ]:
# Does difficulty track design complexity? Metrics sliced by sketch count,
# with truncated designs shown as their own bucket.
ok_df["bucket"] = np.where(
    ok_df["truncated"], "truncated",
    ok_df["n_sketches"].clip(upper=6).astype(int).astype(str),
)
order = sorted(ok_df["bucket"].unique(), key=lambda b: (b == "truncated", b))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, metric in zip(axes, ["chamfer", "iou"]):
    groups = [ok_df.loc[ok_df["bucket"] == b, metric].dropna() for b in order]
    box = ax.boxplot(groups, tick_labels=order, patch_artist=True)
    for patch, color in zip(box["boxes"], plt.get_cmap("tab10").colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_xlabel("sketches per design")
    ax.set_ylabel(metric)
    ax.grid(alpha=0.3, axis="y")
fig.suptitle("reconstruction quality vs design complexity")
fig.tight_layout()
plt.show()

In [ ]:
# Best, median, and worst by Chamfer: sketches in, ground truth, and the
# reconstruction, one row each.
ranked = ok_df.sort_values("chamfer").reset_index(drop=True)
picks = [
    ("best", ranked.iloc[0]["design_id"]),
    ("median", ranked.iloc[len(ranked) // 2]["design_id"]),
    ("worst", ranked.iloc[-1]["design_id"]),
]
fig, axes = plt.subplots(3, 3, figsize=(10.5, 9.6))
for row_i, (label, design_id) in enumerate(picks):
    item = test_dataset[test_dataset.ids.index(design_id)]
    with torch.no_grad():
        code = smodel.encode_sketches(model, item["sketch"][None].to(device))[0].float()
    pred_mesh = seval.extract_mesh(model, code, res=MC_RES, threshold=cfg.mc_threshold)
    axes[row_i, 0].imshow(item["stack_uint8"][0], cmap="gray", vmin=0, vmax=255)
    axes[row_i, 1].imshow(item["views"][0])
    if pred_mesh is not None:
        pred_rgb, _ = gt_renderer.render_mesh(pred_mesh, [item["c2ws"][0]])
        axes[row_i, 2].imshow(pred_rgb[0])
    cd = float(ranked[ranked["design_id"] == design_id]["chamfer"].iloc[0])
    axes[row_i, 0].set_ylabel(f"{label}\nCD {cd:.3f}", fontsize=9)
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
axes[0, 0].set_title("first sketch")
axes[0, 1].set_title("ground truth")
axes[0, 2].set_title("reconstruction")
fig.tight_layout()
plt.show()

In [ ]:
# Turntable GIFs for the same three picks - the shareable artifact.
from IPython.display import Image as IPImage
from IPython.display import display as ipy_display

orbit_c2ws = [
    cameras.build_spherical_c2w(az, 20.0, CAMERA_DISTANCE)
    for az in np.linspace(0.0, 360.0, 36, endpoint=False)
]
for label, design_id in picks:
    item = test_dataset[test_dataset.ids.index(design_id)]
    with torch.no_grad():
        code = smodel.encode_sketches(model, item["sketch"][None].to(device))[0].float()
    pred_mesh = seval.extract_mesh(model, code, res=MC_RES, threshold=cfg.mc_threshold)
    if pred_mesh is None:
        continue
    frames, _ = gt_renderer.render_mesh(pred_mesh, orbit_c2ws)
    gif_path = rdir / "eval" / f"turntable_{label}_{design_id}.gif"
    viz.turntable_gif(list(frames), gif_path)
    print(f"{label}: {design_id}")
    ipy_display(IPImage(data=gif_path.read_bytes(), format="gif"))

In [ ]:
# Zero-shot baseline: the pretrained RGB TripoSR fed each design's dataset
# thumbnail (a rendered photo of the finished part). Same designs, same
# metrics, resumable CSV.
baseline_path = rdir / "eval" / f"metrics_baseline_{SPLIT_PREFIX}test.csv"
if BASELINE_THUMBNAIL:
    baseline_done = (
        set(pd.read_csv(baseline_path)["design_id"]) if baseline_path.exists() else set()
    )
    baseline_ids = [d for d in evaluated_ids if d not in baseline_done]
    if baseline_ids:
        print("loading pretrained RGB TripoSR for the baseline...")
        baseline_model = smodel.load_pretrained_tsr(TRIPOSR_DIR, device=device)
        baseline_model.renderer.set_chunk_size(cfg.eval_chunk)
        for design_id in tqdm(baseline_ids, desc="baseline"):
            row = {"design_id": design_id}
            try:
                png_path = scolab.extract_members(
                    zip_path, [zip_index[design_id]["png"]], WORK_DIR
                )[0]
                thumb = Image.open(png_path).convert("RGB")
                with torch.no_grad():
                    codes = baseline_model([thumb], device=device)
                pred_mesh = seval.extract_mesh(
                    baseline_model, codes[0], res=MC_RES, threshold=cfg.mc_threshold
                )
                gt_mesh = load_gt_mesh(design_id)
                if pred_mesh is None or gt_mesh is None:
                    row["failed"] = True
                else:
                    p_gt = seval.sample_surface_points(gt_mesh, cfg.n_points, seed=SEED)
                    p_pred = seval.sample_surface_points(pred_mesh, cfg.n_points, seed=SEED)
                    row["chamfer"] = seval.chamfer_distance(p_gt, p_pred)
                    for tau in cfg.taus:
                        row[f"f@{tau}"] = seval.fscore(p_pred, p_gt, tau)[0]
                    row["iou"] = seval.voxel_iou(
                        seval.occupancy_grid(p_gt, cfg.iou_res),
                        seval.occupancy_grid(p_pred, cfg.iou_res),
                    )
            except Exception as error:
                row["failed"] = True
                row["error"] = repr(error)
            pd.DataFrame([row]).to_csv(
                baseline_path, mode="a", header=not baseline_path.exists(), index=False
            )
        del baseline_model
        if device == "cuda":
            torch.cuda.empty_cache()
    print("baseline rows:", len(pd.read_csv(baseline_path)) if baseline_path.exists() else 0)
else:
    print("baseline disabled")

In [ ]:
# Ours vs baseline, side by side on the designs both evaluated.
baseline_df = pd.read_csv(baseline_path) if baseline_path.exists() else pd.DataFrame()
if not baseline_df.empty:
    shared = set(ok_df["design_id"]) & set(baseline_df["design_id"])
    ours = ok_df[ok_df["design_id"].isin(shared)]
    base = baseline_df[baseline_df["design_id"].isin(shared)]
    metrics = ["chamfer", "f@0.02", "iou"]
    ours_med = [ours[m].median() for m in metrics]
    base_med = [base[m].median() for m in metrics]
    x = np.arange(len(metrics))
    colors = plt.get_cmap("tab10").colors
    fig, ax = plt.subplots(figsize=(7.5, 3.4))
    ax.bar(x - 0.18, ours_med, width=0.36, color=colors[0], label="sketches (ours)")
    ax.bar(x + 0.18, base_med, width=0.36, color=colors[1], label="thumbnail (baseline)")
    ax.set_xticks(x, metrics)
    ax.set_title(f"median metrics on {len(shared)} shared designs "
                 "(chamfer lower is better; f-score and iou higher)")
    ax.legend()
    plt.show()
else:
    print("no baseline results - enable BASELINE_THUMBNAIL and re-run")

In [ ]:
# Failure board: the worst designs by Chamfer, annotated with the
# properties that might explain them.
worst = ok_df.sort_values("chamfer", ascending=False).head(8)
fig, axes = plt.subplots(2, 4, figsize=(13, 6.4))
for ax, (_, row) in zip(axes.ravel(), worst.iterrows()):
    item = test_dataset[test_dataset.ids.index(row["design_id"])]
    ax.imshow(item["stack_uint8"][0], cmap="gray", vmin=0, vmax=255)
    ax.set_title(
        f"{row['design_id'][:12]}\nCD {row['chamfer']:.3f}  "
        f"sk {int(row['n_sketches'])}  trunc {bool(row['truncated'])}",
        fontsize=8,
    )
    ax.axis("off")
fig.suptitle("worst reconstructions - first sketch channel shown")
fig.tight_layout()
plt.show()

n_failed = int(results_df.get("failed", pd.Series(dtype=bool)).fillna(False).sum())
print(f"designs with no extractable surface at all: {n_failed}")

## Findings

Fill this in from the cells above after a full evaluation run:

- overall quality: median Chamfer distance and F-score at the middle threshold, and what the median-quality gallery rows actually look like;
- where it works: which kinds of parts (few sketches, simple extrudes) score best;
- where it struggles: what the failure board has in common (many sketches, truncation, tiny features, thin walls);
- against the baseline: does sketch-driven reconstruction beat the pretrained model looking at a rendered photo of the part? Either answer is a real finding - the baseline sees shading and silhouette, ours sees the design's actual construction curves;
- next experiments: the first ablations worth running (LoRA rank, training the triplane tokens, per-sketch normalization, more steps).